In [1]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage
import base64

load_dotenv()

d:\Agentic-Ai\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
# encode the image and send to the vision model

with open("blood_report.png","rb") as f:
  image_b64 = base64.b64encode(f.read()).decode()

image_b64[:200]

'iVBORw0KGgoAAAANSUhEUgAAAdMAAAIxCAYAAAAWmlK5AAAAAXNSR0IArs4c6QAAAARnQU1BAACxjwv8YQUAAAAJcEhZcwAAEnQAABJ0Ad5mH3gAAP+lSURBVHhe7J11nFXV+v/fa+epKbpTBRUVVEywWzDAFrx26/Xa3djdDSp2NxYWCijd3c0weXLX+v2x9hmGw6gg'

In [3]:
llm = ChatGroq(
  model="qwen/qwen3.6-27b"
)


message = HumanMessage(content=[
    {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_b64}"}},
    {"type": "text",      "text": "This is a blood work report. Extract all test results and flag any values outside the normal range."}
])

response = llm.invoke([message])
print(response.content)


<think>
The user wants me to extract data from a blood test report image.
I need to:
1.  **Identify the sections:** Complete Blood Count (CBC), Lipid Panel, Metabolic Panel, Liver Function.
2.  **Extract the data:** For each test, I need the Test Name, Result, Reference Range, and Unit.
3.  **Flag abnormalities:** Compare the Result to the Reference Range. If it's outside, mark it as high or low.

**Step 1: Analyze the image sections.**

*   **Header:** Narmada Diagnostics & Research Centre, Ahmedabad. Patient: Rajesh Sharma. Date: May 7, 2026.
*   **Complete Blood Count (CBC):**
    *   Hemoglobin: Result 15.1, Range 13.5-17.5. (Normal)
    *   Hematocrit: Result 44, Range 41-53. (Normal)
    *   WBC Count: Result 6.8, Range 4.5-11.0. (Normal)
    *   Platelet Count: Result 220, Range 150-400. (Normal)
*   **Lipid Panel:**
    *   Total Cholesterol: Result 238, Range <200. (High - flagged in yellow)
    *   LDL Cholesterol (Calc.): Result 162, Range <100. (High - flagged in yellow)
 

In [ ]:
# make the diet recommendation agent! (task!!!!)

In [4]:
from langchain.tools import tool
from langchain.agents import create_agent

In [5]:
@tool
def get_diet_recommendation(condition: str) -> dict:
    """Given a health condition, returns a diet plan. Condition must be one of: normal, high_cholesterol, high_sugar."""
    diet_plans = {
        "high_cholesterol": {
            "eat":        ["fruits", "vegetables", "whole grains", "lean protein"],
            "do_not_eat": ["red meat", "fried food", "full-fat dairy", "processed snacks"],
        },
        "high_sugar": {
            "eat":        ["vegetables", "whole grains", "legumes", "nuts"],
            "do_not_eat": ["white rice", "white sugar", "junk food", "sugary drinks"],
        },
        "normal": {
            "eat":        ["vegetables", "fruits", "whole grains", "lean protein"],
            "do_not_eat": ["excessive sugar", "processed food", "trans fats"],
        },
    }
    return diet_plans.get(condition, diet_plans["normal"])

In [6]:
SYSTEM_PROMPT = """
You are a helpful medical and nutrition assistant.
For the input blood work image, extract the numbers and the normal range, then categorize
the condition as one of: normal, high_cholesterol, high_sugar.
Then call the appropriate tool to retrieve and present the diet plan.
"""

diet_agent = create_agent(
    llm,
    tools=[get_diet_recommendation],
    system_prompt=SYSTEM_PROMPT,
)

In [7]:
result = diet_agent.invoke({
    "messages": [HumanMessage(content=[
        {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_b64}"}},
        {"type": "text",      "text": "Analyse this blood work report and suggest a diet plan."},
    ])]
})

print(result["messages"][-1].content)

Based on the blood work report, the patient has elevated cholesterol levels (Total Cholesterol: 238 mg/dL, LDL: 162 mg/dL, Triglycerides: 188 mg/dL).

Here is the recommended diet plan for managing high cholesterol:

**Recommended Foods:**
*   Fruits
*   Vegetables
*   Whole grains
*   Lean protein

**Foods to Avoid:**
*   Red meat
*   Fried food
*   Full-fat dairy
*   Processed snacks
